# 데이터 입수

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("residentmario/ramen-ratings")

print("Path to dataset files:", path)

# 세팅구역

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="icefire", style="whitegrid", font_scale=1)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Tangba 12'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
ramen_df = pd.read_csv(f'{path}/ramen-ratings.csv')

# 데이터 정보 확인
## .info()

In [ ]:
ramen_df.info()

## .describe()

In [ ]:
ramen_df.describe()

In [ ]:
ramen_df.describe(include='O')

## .isna().sum()

In [ ]:
ramen_df.isna().sum()

## .columns

In [ ]:
ramen_df.columns

## .head()

In [ ]:
ramen_df.head()

# 전처리
## 결측값 확인

In [ ]:
ramen_df.query('Style.isna()')

In [ ]:
ramen_df['Style'] = ramen_df['Style'].fillna('Pack')
# 둘다 팩이래요

## 니들 숫잔데 왜 문자인 척 하냐

In [ ]:
ramen_df['Stars'] = pd.to_numeric(ramen_df['Stars'], errors='coerce') # 별점
ramen_df['Review #'] = pd.to_numeric(ramen_df['Review #'], errors='coerce') # 리뷰 수

## 브랜드 이름 통일
- Chorip Dong = ChoripDong이더군요. 처음 보는 브랜드라고요? 저도 그래요. 하지만 재외동포들은 많이 봤을거임.

In [ ]:
ramen_df['Brand'] = ramen_df['Brand'].replace('Chorip Dong', 'ChoripDong')
ramen_df['Brand'] = ramen_df['Brand'].replace('Samyang Foods', 'Samyang')

## 아니 줄바꿈 문자가 왜있는데!!!

In [ ]:
ramen_df['Top Ten'] = ramen_df['Top Ten'].replace('\n', np.nan, regex=True)

# 분석 렛츄고
- 어지간한건 다 묶여있어서 별도로 묶을게 없습니다...

## 국가별 라면 개수

In [ ]:
ramen_df_count = ramen_df.groupby('Country')['Review #'].agg('count').sort_values(ascending = False).reset_index()

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(ramen_df_count, x = 'Country', y = 'Review #', hue = 'Country', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('국가별 라면 개수', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('라면 개수')
plt.xticks(rotation=90)
plt.show()

### Rating이 3 이상인 라면만

In [ ]:
ramen_df_rating3 = ramen_df.query('Stars >= 3') # 별점 3점 이상
ramen_df_rating3 = ramen_df_rating3.groupby('Country')['Review #'].agg('count').sort_values(ascending = False).reset_index()

ramen_df_rating3['Review #']

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(ramen_df_rating3, x = 'Country', y = 'Review #', hue = 'Country', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('국가별 라면 개수 (별점 3점 이상)', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('라면 개수')
plt.xticks(rotation=90)
plt.show()

## 국가별 별점 평균

In [ ]:
ramen_df_mean = ramen_df.groupby('Country')['Stars'].agg('mean').sort_values(ascending = False).reset_index()
ramen_df_mean

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(ramen_df_mean, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.xlabel('국가')
plt.ylabel('별점 평균')
plt.xticks(rotation=90)
plt.title('국가별 라면 별점 평균', fontsize = 20)
plt.show()

- 설마 불닭볶음면이 너무 매워서...? ~~그맛에 먹는거 아니었나~~

## 국가별로 별점이 제일 높은 라면은?

In [ ]:
high_star_idx = ramen_df.dropna(subset=['Country', 'Stars']).groupby('Country')['Stars'].idxmax()
ramen_df.loc[high_star_idx].sort_values('Stars', ascending = False).reset_index()

## K-라면의 위상은?

In [ ]:
k_ramen = ramen_df.query('Country == "South Korea"') # 니네 DB에 이북산 라면도 있니?
k_ramen

In [ ]:
k_ramen_cnt = k_ramen.groupby('Brand')['Stars'].agg('count').sort_values(ascending = False).reset_index()# 35개나 있어? ㄷㄷ

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(k_ramen_cnt, x = 'Brand', y = 'Stars', hue = 'Brand', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('국가별 라면 개수', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('라면 개수')
plt.xticks(rotation=90)
plt.show()

- 저 초립동이 브랜드 처음 보시죠? 근데 외국 한인마트에서 많이 보이는 브랜드랍니다.

## K-라면 평균 별점

In [ ]:
k_ramen_mean = k_ramen.groupby('Brand')['Stars'].agg('mean').sort_values(ascending = False).reset_index()

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(k_ramen_mean, x = 'Brand', y = 'Stars', hue = 'Brand', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('국가별 라면 별점', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('라면 개수')
plt.xticks(rotation=90)
plt.show()

## K-라면 TOP 5 브랜드
### 팔도

In [ ]:
paldo = k_ramen.query('Brand == "Paldo"')
paldo # 불닭 언제 나오나 본다 내가

#### 팔도 별점 5점짜리는 어떤 라면인가?

In [ ]:
paldo_5_star = paldo.query('Stars >= 5')
paldo_5_star

- 그... 꼬꼬면을 제가 참 좋아하긴 해요... 그렇게 맵지도 않고 또 칼칼함은 지대로거든...
- 근데 이게 그정도야?

#### 도시락
- 삼양 미안... 불닭 니네껀데 깜빡했어...

In [ ]:
target_ramens = paldo[paldo['Variety'].str.contains('Dosirac', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

### 농심
- 저는 여기 라면은 새우탕면만 먹습니다.
- 그게 백목이버섯 넣어서 먹으면 맛있음.

In [ ]:
nongshim = k_ramen.query('Brand == "Nongshim"')
nongshim # 불닭 언제 나오나 본다 내가

#### 농심 5점짜리 빠밤

In [ ]:
nongshim_5_star = nongshim.query('Stars >= 5')
nongshim_5_star

#### 쫄깃쫄깃 오동통통 농심 너구리~
- 전 순한맛이요.

In [ ]:
target_ramens = nongshim[nongshim['Variety'].str.contains('Neoguri', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

#### 신라면
- 전 매워서 못먹지만.

In [ ]:
target_ramens = nongshim[nongshim['Variety'].str.contains('Shin', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

#### 안성탕면
- 니들 빨리 와서 순하리 먹어보고 가라

In [ ]:
target_ramens = nongshim[nongshim['Variety'].str.contains('Ansungtangmyun', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

### 삼양
- 불닭 거기 맞습니다.

In [ ]:
samyang = k_ramen.query('Brand == "Samyang"')
samyang # 불닭 언제 나오나 본다 내가

#### 5점짜리 두둥

In [ ]:
samyang_5_star = samyang.query('Stars >= 5')
samyang_5_star

#### 불닭볶음며어언
- 월드와이드 스파이시 (본인 못먹음)

In [ ]:
target_ramens = samyang[samyang['Variety'].str.contains('Buldak', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

#### 삼양라면(류)

In [ ]:
target_ramens = samyang[samyang['Variety'].str.contains('Samyang Ramen|Samyang Ramyun', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

### 오뚜기

In [ ]:
ottogi = k_ramen.query('Brand == "Ottogi"')
ottogi

#### 오뚜기 5스타

In [ ]:
ottogi_5_star = ottogi.query('Stars >= 5')
ottogi_5_star

#### 진라면
- 저는 순한맛만 먹어요...

In [ ]:
target_ramens = ottogi[ottogi['Variety'].str.contains('Jin Ramen', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

##### 진순 vs 진매

In [ ]:
jin_mild = ottogi[ottogi['Variety'].str.contains('Mild', case=False)]
jin_hot = ottogi[ottogi['Variety'].str.contains('Hot', case=False)]

mean_mild = np.mean(jin_mild['Stars'])
mean_hot = np.mean(jin_hot['Stars'])

print(f'진순이 별점: {mean_mild} | 진매 별점 {mean_hot}')
if mean_mild > mean_hot:
    print('진순이 만세!')
else:
    print('진순이 매니아는 웁니다. ')

#### 솔직히 참꺠라면 별점 궁금했어요

In [ ]:
target_ramens = ottogi[ottogi['Variety'].str.contains('Sesame', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

#### 오동통

In [ ]:
target_ramens = ottogi[ottogi['Variety'].str.contains('Odongtong', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

#### 뿌셔뿌셔...는...스낵인디...

In [ ]:
target_ramens = ottogi[ottogi['Variety'].str.contains('Ppushu', case=False)]
print(target_ramens[['Brand', 'Variety', 'Stars']])

### 풀무원
- 로스팅 시리즈 제가 좋아합니다.

In [ ]:
pulmuone = k_ramen.query('Brand == "Pulmuone"')
pulmuone # 불닭 언제 나오나 본다 내가

#### 풀무원의 파이브 스타즈

In [ ]:
pulmuone_5_star = pulmuone.query('Stars >= 5')
pulmuone_5_star

## K-라면중에 TOP 10에 든 라면도 있나요?

In [ ]:
k_ramen.query('not `Top Ten`.isna()')

- \n은 왜 저기 껴있는겨... 아오...

## TOP 10에 제일 많이 들어간 국가는?

In [ ]:
top10_nominated = ramen_df.query('not `Top Ten`.isna()')
top10_nominated_cnt = top10_nominated.groupby('Country')['Variety'].agg('count').sort_values(ascending = False).reset_index()

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(top10_nominated_cnt, x = 'Country', y = 'Variety', hue = 'Country', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('국가별 Top 10에 입성한 개수', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('입성한 라면')
plt.xticks(rotation=90)
plt.show()

### 의 별점 평균

In [ ]:
top10_nominated_mean = top10_nominated.groupby('Country')['Stars'].agg('mean').sort_values(ascending = False).reset_index()

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(top10_nominated_mean, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('국가별 Top 10에 입성한 라면들의 별점 평균', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('입성한 라면들의 별점 평균')
plt.xticks(rotation=90)
plt.show()

## Star가 1 미만인 라면도 있나요?

In [ ]:
ramen_df.query("Stars < 1")

In [ ]:
ramen_under_1 = ramen_df.query("Stars < 1").groupby('Country')['Stars'].agg('count').sort_values(ascending=False).reset_index()

In [ ]:
plt.figure(figsize = (18, 9))
ax = sns.barplot(ramen_under_1, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral')

# 라벨 박아줘야죠
for i in ax.patches:
    height = i.get_height()
    ax.annotate(f'{height:.1f}',  # 표시할 텍스트 (소수점 1자리)
                (i.get_x() + i.get_width() / 2., height), # 위치: 막대 중앙 상단
                ha='center', va='bottom', size=11) # 정렬 및 크기

plt.title('별점 1점 미만인 라면들의 국가 분포', fontsize = 20)
plt.xlabel('국가')
plt.ylabel('1점 미만인 라면 개수')
plt.xticks(rotation=90)
plt.show()

## 컵라면 그리고 봉지라면

In [ ]:
style_cup = ['Cup', 'Bowl']
style_pack = ['Pack']

ramen_cup = ramen_df.query('Style in @style_cup')
ramen_pack = ramen_df.query('Style in @style_pack')

### 각 패키지당 별점 5점의 개수 빠밤

In [ ]:
cup_5star = ramen_cup.query('Stars >= 5')
pack_5star = ramen_pack.query('Stars >= 5')

print(f'5-star rated cup: {cup_5star.shape[0]} | pack rated pack: {pack_5star.shape[0]}')

- 다들... 컵라면 잘 안 드세요?

### 각 패키지당 별점 1점 미만의 개수

In [ ]:
cup_1star = ramen_cup.query('Stars < 1')
pack_1star = ramen_pack.query('Stars < 1')

print(f'5-star rated cup: {cup_1star.shape[0]} | pack rated pack: {pack_1star.shape[0]}')

### 비율로 매겨보자고!

In [ ]:
# 컵라면(Cup, Bowl) 비율 계산
cup_counts = ramen_cup['Stars'].value_counts(normalize=True)
cup_5 = cup_counts.get(5, 0) * 100
cup_under_1 = cup_counts[cup_counts.index < 1].sum() * 100

# 봉지라면(Pack) 비율 계산
pack_counts = ramen_pack['Stars'].value_counts(normalize=True)
pack_5 = pack_counts.get(5, 0) * 100
pack_under_1 = pack_counts[pack_counts.index < 1].sum() * 100

# 결과 출력
print(f"[컵라면] 5점 비율: {cup_5:.2f}%, 1점 미만 비율: {cup_under_1:.2f}%")
print(f"[봉지라면] 5점 비율: {pack_5:.2f}%, 1점 미만 비율: {pack_under_1:.2f}%")

### 봉지면 vs 컵라면: 5점짜리 국가 분포

In [ ]:
cup_5_cnt = cup_5star.groupby('Country')['Stars'].agg('count').sort_values(ascending=False).reset_index() # 컵
pack_5_cnt = pack_5star.groupby('Country')['Stars'].agg('count').sort_values(ascending=False).reset_index() # 팩

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(20, 10))

ax[0] = sns.barplot(cup_5_cnt, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral', ax=ax[0], legend=False)
ax[0].set_title('컵라면 별점 5점짜리 국가 분포', fontsize = 20)
ax[0].set_xlabel('국가')
ax[0].set_ylabel('개수')
ax[0].tick_params(axis='x', rotation=90)

ax[1] = sns.barplot(pack_5_cnt, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral', ax=ax[1], legend=False)
ax[1].set_title('봉지라면 별점 5점짜리 국가 분포', fontsize = 20)
ax[1].set_xlabel('국가')
ax[1].set_ylabel('개수')
ax[1].tick_params(axis='x', rotation=90)

# 레이아웃 조정 후 출력
plt.tight_layout()
plt.show()

### 봉지면 vs 컵라면: 1점짜리 국가 분포

In [ ]:
cup_1_cnt = cup_1star.groupby('Country')['Stars'].agg('count').sort_values(ascending=False).reset_index() # 컵
pack_1_cnt = pack_1star.groupby('Country')['Stars'].agg('count').sort_values(ascending=False).reset_index() # 팩

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(20, 10))

ax[0] = sns.barplot(cup_1_cnt, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral', ax=ax[0], legend=False)
ax[0].set_title('컵라면 별점 1점짜리 국가 분포', fontsize = 20)
ax[0].set_xlabel('국가')
ax[0].set_ylabel('개수')
ax[0].tick_params(axis='x', rotation=90)

ax[1] = sns.barplot(pack_1_cnt, x = 'Country', y = 'Stars', hue = 'Country', palette = 'Spectral', ax=ax[1], legend=False)
ax[1].set_title('봉지라면 별점 1점짜리 국가 분포', fontsize = 20)
ax[1].set_xlabel('국가')
ax[1].set_ylabel('개수')
ax[1].tick_params(axis='x', rotation=90)

# 레이아웃 조정 후 출력
plt.tight_layout()
plt.show()

### 봉지면 vs 컵라면: K-라면의 비중

In [ ]:
# 국가가 사우쓰 코리아냐 아니냐로 구별
# 람다식 쓰시져
ramen_cup_kor = ramen_cup.copy()
ramen_cup_kor['Country'] = ramen_cup['Country'].apply(lambda x:"K-ramyeon" if x == "South Korea" else "Other")

ramen_pack_kor = ramen_pack.copy()
ramen_pack_kor['Country'] = ramen_pack['Country'].apply(lambda x:"K-ramyeon" if x == "South Korea" else "Other")

In [ ]:
# 1. 데이터 집계 (Cup & Pack 각각)
cup_counts = ramen_cup_kor['Country'].value_counts()
pack_counts = ramen_pack_kor['Country'].value_counts()

# 2. 파이차트 그리기 (1행 2열)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

# 공통 스타일 설정
colors = ['#ff9999', '#ffc000'] # K-ramyeon은 눈에 띄게!
explode = [0.1, 0] # K-ramyeon 살짝 튀어나오게

# 컵라면 파이차트
ax[0].pie(cup_counts, labels=cup_counts.index, autopct='%1.1f%%',
          startangle=90, colors=colors, explode=explode, shadow=True)
ax[0].set_title('컵라면: 한국 vs 타국가 비율', fontsize=16)

# 봉지라면 파이차트
ax[1].pie(pack_counts, labels=pack_counts.index, autopct='%1.1f%%',
          startangle=90, colors=colors, explode=explode, shadow=True)
ax[1].set_title('봉지라면: 한국 vs 타국가 비율', fontsize=16)

plt.tight_layout()
plt.show()

### 봉지면 vs 컵라면: TOP 10에 든 라면은 몇 개인가?

In [ ]:
cup_top10 = ramen_cup.copy()
pack_top10 = ramen_pack.copy()
cup_top10['Is Top Ten'] = cup_top10['Top Ten'].apply(lambda x: 'Top Ten' if pd.notnull(x) and x != '' else 'None')
pack_top10['Is Top Ten'] = pack_top10['Top Ten'].apply(lambda x: 'Top Ten' if pd.notnull(x) and x != '' else 'None')

In [ ]:
# 1. 데이터 집계 (Cup & Pack 각각)
cup_counts = cup_top10['Is Top Ten'].value_counts()
pack_counts = pack_top10['Is Top Ten'].value_counts()

# 2. 파이차트 그리기 (1행 2열)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

# 공통 스타일 설정
colors = ['#ff9999', '#ffc000'] # K-ramyeon은 눈에 띄게!
explode = [0.1, 0] # K-ramyeon 살짝 튀어나오게

# 컵라면 파이차트
ax[0].pie(cup_counts, labels=cup_counts.index, autopct='%1.1f%%',
          startangle=90, colors=colors, shadow=True)
ax[0].set_title('컵라면: TOP 10 비율', fontsize=16)

# 봉지라면 파이차트
ax[1].pie(pack_counts, labels=pack_counts.index, autopct='%1.1f%%',
          startangle=90, colors=colors, explode=explode, shadow=True)
ax[1].set_title('봉지라면: TOP 10 비율', fontsize=16)

plt.tight_layout()
plt.show()

#### K-라면은 있는가 (컵)

In [ ]:
# 컵라면 중 Top Ten에 선정된 녀석들만 추출
cup_top10_winners = cup_top10.query('`Is Top Ten` == "Top Ten"')

# 그 안에서 국가별 비중 확인
cup_top10_korea = cup_top10_winners['Country'].value_counts()
print("Top 10에 선정된 컵라면들의 국적 분포:")
print(cup_top10_korea)

In [ ]:
# 컵라면 중 Top Ten에 선정된 제품의 브랜드와 제품명 출력
elite_cup = cup_top10.query('`Is Top Ten` == "Top Ten"')[['Brand', 'Variety', 'Country', 'Top Ten']]

print("--- 컵라면계의 전설 2인조 ---")
print(elite_cup)

- 아 줄바꿈 문자를 정리했더니 사라졌... 

#### K-라면은 있는가 (봉지)

In [ ]:
# 컵라면 중 Top Ten에 선정된 녀석들만 추출
pack_top10_winners = pack_top10.query('`Is Top Ten` == "Top Ten"')

# 그 안에서 국가별 비중 확인
pack_top10_korea = pack_top10_winners['Country'].value_counts()
print("Top 10에 선정된 봉지라면들의 국적 분포:")
print(pack_top10_korea)

In [ ]:
# 컵라면 중 Top Ten에 선정된 제품의 브랜드와 제품명 출력
elite_pack = pack_top10.query('`Is Top Ten` == "Top Ten" and Country == "South Korea"')[['Brand', 'Variety', 'Country', 'Top Ten']]

print("--- 봉지라면계의 전설(들) ---")
print(elite_pack)

- 경규옹... 당신은 대체 무엇을 만든것입니까...